# 01 — Train the temporal model (stacked RNN)

Set `CONDITION` below and run the whole notebook. Run it once with `"po"` and
once with `"raw"`; each run writes `results/rnn_<condition>.json`, which is all
the results notebook needs. Nothing here has to be re-run unless the model or
the features change.

In [5]:
CONDITION = "raw"          # "po" = participant-only clips, "raw" = full interview recordings

import common
from common import (SEGMENT_LENGTHS, SEEDS, load_metadata, load_features,
                    run_rnn_seeds, save_results, device)

print("Condition:", CONDITION)
print("Features :", common.FEATURE_PATHS[CONDITION])
print("Device   :", device)
print("Seeds    :", SEEDS)

Condition: raw
Features : ./androids_is09_02.npz
Device   : cuda
Seeds    : (0, 1, 2, 3, 4)


In [6]:
interview_df, label_of, gender_of, interview_folds = load_metadata()
data = load_features(CONDITION)

print(f"Speakers: {len(data)} | labelled: {len(label_of)}")
print("Folds:", {k: len(v) for k, v in interview_folds.items()})

Speakers: 116 | labelled: 116
Folds: {1: 24, 2: 23, 3: 23, 4: 23, 5: 23}


## Training

For every segment length the cross-validation is repeated `R = 5` times with
different random initialisations, and the repetitions are averaged **within
each fold**. The stored metric lists therefore hold one entry per fold.

In [7]:
results = {}

for seg_len in SEGMENT_LENGTHS:
    print(f"\n=== segment length {seg_len} ===")

    smv_folds, wa_folds, loss_curves, pooled = run_rnn_seeds(
        data, label_of, interview_folds, seg_len,
        seeds=SEEDS, hidden_size=70, epochs=30
    )

    results[str(seg_len)] = {
        "smv": smv_folds,
        "wa": wa_folds,
        "pooled": pooled,
        "loss_curves": {str(k): v for k, v in loss_curves.items()},   # seed 0 only
    }

    import numpy as np
    for name, folds in [("SMV", smv_folds), ("WA", wa_folds)]:
        f1 = np.array([f["f1"] for f in folds])
        acc = np.array([f["acc"] for f in folds])
        print(f"  {name}: acc {acc.mean()*100:.1f} ± {acc.std(ddof=1)*100:.1f}   "
              f"F1 {f1.mean()*100:.1f} ± {f1.std(ddof=1)*100:.1f}")


=== segment length 32 ===
  SMV: acc 77.2 ± 3.8   F1 74.7 ± 3.3
  WA: acc 77.4 ± 2.6   F1 75.1 ± 4.8

=== segment length 64 ===
  SMV: acc 72.4 ± 3.6   F1 66.9 ± 6.2
  WA: acc 72.7 ± 2.2   F1 69.0 ± 7.1

=== segment length 128 ===
  SMV: acc 69.7 ± 2.8   F1 65.7 ± 9.9
  WA: acc 70.7 ± 2.6   F1 68.2 ± 8.4

=== segment length 256 ===
  SMV: acc 70.3 ± 2.8   F1 66.8 ± 8.9
  WA: acc 70.7 ± 3.3   F1 67.4 ± 9.3

=== segment length 512 ===
  SMV: acc 63.2 ± 7.5   F1 59.3 ± 4.6
  WA: acc 64.4 ± 8.7   F1 62.0 ± 8.3

=== segment length 1024 ===
  SMV: acc 51.0 ± 14.7   F1 42.3 ± 17.8
  WA: acc 53.8 ± 14.6   F1 48.3 ± 17.1


In [8]:
save_results({
    "condition": CONDITION,
    "model": "rnn",
    "seeds": list(SEEDS),
    "segment_lengths": SEGMENT_LENGTHS,
    "hidden_size": 70,
    "num_layers": 2,
    "epochs": 30,
    "results": results,
}, f"rnn_{CONDITION}")

Saved results\rnn_raw.json


'results\\rnn_raw.json'